# מפגש 20 — הדגמה מודרכת Audited
סיפור: מתוך 100 הודעות 5 ספאם ו־95 תקינות. מסנן תמיד OK נראה מוצלח לפי Accuracy, אבל האם הוא מגן מפני ספאם? **חיובי=Spam**. הנתונים מומצאים; אין להשתמש בהם כהוכחת ביצועים על אוכלוסייה חדשה.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.dummy import DummyClassifier
from sklearn.metrics import confusion_matrix,accuracy_score,precision_score,recall_score,f1_score

## לפני הקוד: מטריצה ידנית
כתבו TN, FP, FN ו־TP. שורות הן אמת ועמודות הן תחזית. מה סכום כל התאים?

In [ ]:
actual=["OK"]*95+["Spam"]*5
predicted=["OK"]*100
labels=["OK","Spam"]
cm=confusion_matrix(actual,predicted,labels=labels)
table=pd.DataFrame(cm,index=labels,columns=labels)
display(table)

In [ ]:
assert len(actual)==len(predicted)==100
assert cm.tolist()==[[95,0],[5,0]]
assert cm.shape==(2,2)
assert cm.sum()==100
tn,fp,fn,tp=cm.ravel()
assert (tn,fp,fn,tp)==(95,0,5,0)
assert tn+fp==95
assert fn+tp==5

## Accuracy ו־Recall: שני מכנים שונים
Accuracy=(TP+TN)/100. Recall=TP/(TP+FN). Precision=TP/(TP+FP): כאשר אין חיזויים חיוביים, 0/0 אינו מוגדר מתמטית.

In [ ]:
accuracy=(tp+tn)/cm.sum()
recall=tp/(tp+fn)
assert accuracy==.95
assert recall==0
assert accuracy_score(actual,predicted)==accuracy
assert recall_score(actual,predicted,pos_label="Spam")==recall
assert precision_score(actual,predicted,pos_label="Spam",zero_division=0)==0
print("Accuracy",accuracy,"Recall",recall,"Precision reporting policy",0)

## בונים קו בסיס בעזרת DummyClassifier
strategy='most_frequent' משתמש בתווית הנפוצה שנמצאה ב־Train. אל תבחרו כלל לפי Test. הנתונים כאן מוצגים כהדגמה סינתטית בלבד.

In [ ]:
X_train=np.arange(100).reshape(-1,1)
baseline=DummyClassifier(strategy="most_frequent")
baseline.fit(X_train,actual)
predict_baseline=baseline.predict(X_train)
assert set(predict_baseline)=={"OK"}
assert list(predict_baseline)==predicted
assert accuracy_score(actual,predict_baseline)==.95
assert recall_score(actual,predict_baseline,pos_label="Spam")==0
print(baseline.classes_,baseline.class_prior_)

## ניסוי שינוי שכיחות
בדוגמה חדשה עם 99 OK וספאם אחד, אם הכלל עדיין מנבא OK — מה יקרה ל־Accuracy ומה יקרה ל־Recall? נבקש קודם נימוק.

In [ ]:
actual99=["OK"]*99+["Spam"]
pred99=["OK"]*100
assert accuracy_score(actual99,pred99)==.99
assert recall_score(actual99,pred99,pos_label="Spam")==0
assert accuracy_score(actual99,pred99)>accuracy
assert confusion_matrix(actual99,pred99,labels=labels).tolist()==[[99,0],[1,0]]

## בדיקה חזותית
הציגו שכיחויות והסבירו למה העמודה הנמוכה עלולה להיות החשובה.

In [ ]:
counts=pd.Series(actual).value_counts().reindex(labels)
ax=counts.plot.bar(title="Synthetic class counts")
ax.set_xlabel("True class")
ax.set_ylabel("Messages")
plt.show()
assert counts.sum()==100
assert counts.loc["Spam"]==5

## כרטיס יציאה
האם מאזנים Test באופן אוטומטי? לא: צריך לשקף את השימוש והמטרה. ציינו מדד שהייתם בודקים בנוסף ל־Accuracy, והסבירו מגבלה אחת. בפרויקט 45 דקות מוגנות.